# Batch Normalization — Paper-to-Code Mock (Colab)

**Paper:** Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift (Ioffe & Szegedy, 2015) — https://arxiv.org/abs/1502.03167

Medium mock (~40 min). Fill in the `BatchNorm1d` stub, run the WITH-vs-WITHOUT-BN training demo, then the sanity checks (including a numerical match to `torch.nn.BatchNorm1d`). Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement BatchNorm1d (YOUR TASK)

Input shape `(N, C)`. Train: normalize with the **batch** mean/var (use the BIASED variance, `unbiased=False`), apply learnable `gamma`/`beta`, and update the running buffers (PyTorch updates running var with the UNBIASED variance). Eval: normalize with the **running** stats and update nothing. Respect `self.training`.

In [ ]:
class BatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x):
        # TODO (train): mean = x.mean(0); var = x.var(0, unbiased=False)
        # TODO (train): update running_mean / running_var (use UNBIASED var) under torch.no_grad()
        # TODO (eval):  mean, var = self.running_mean, self.running_var
        # TODO: xhat = (x - mean) / sqrt(var + eps); return gamma * xhat + beta
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (deep net that needs BN to train)
A 6-layer MLP at a learning rate that's too high for the plain net. Without normalization it stalls near the mean predictor; with BN it converges at the SAME learning rate.

In [ ]:
torch.manual_seed(0)
N, D = 512, 20
X = torch.randn(N, D)
w_true = torch.randn(D, 1)
y = torch.sin(X @ w_true) + 0.1 * torch.randn(N, 1)   # nonlinear target

def make_net(use_bn, depth=6, width=128, in_dim=20):
    torch.manual_seed(0)
    layers = [nn.Linear(in_dim, width)]
    if use_bn: layers.append(BatchNorm1d(width))
    layers.append(nn.ReLU())
    for _ in range(depth - 1):
        layers.append(nn.Linear(width, width))
        if use_bn: layers.append(BatchNorm1d(width))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(width, 1))
    return nn.Sequential(*layers)

def train(use_bn, lr=0.05, steps=300):
    torch.manual_seed(0)
    net = make_net(use_bn)
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    net.train()
    for _ in range(steps):
        opt.zero_grad()
        F.mse_loss(net(X), y).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return F.mse_loss(net(X), y).item()

print(f"no-BN final loss = {train(False):.4f}")   # stalls ~0.50 (predicts the mean)
print(f"   BN final loss = {train(True):.6f}")     # converges ~0.00

## 3. Sanity checks

In [ ]:
# 1) train mode normalizes per feature (~0 mean, ~1 var, before gamma/beta)
bn = BatchNorm1d(8).train(); x = torch.randn(1024, 8) * 5 + 3; yv = bn(x)
assert torch.allclose(yv.mean(0), torch.zeros(8), atol=1e-4)
assert torch.allclose(yv.var(0, unbiased=False), torch.ones(8), atol=1e-2)

# 2) eval uses running stats, not batch stats
bn = BatchNorm1d(8).train()
for _ in range(50): bn(torch.randn(256, 8) * 2 + 1)
bn.eval(); xb = torch.randn(256, 8) * 10 + 50; ye = bn(xb)
assert not torch.allclose(ye.mean(0), torch.zeros(8), atol=0.5)
manual = (xb - bn.running_mean) / torch.sqrt(bn.running_var + bn.eps)
assert torch.allclose(ye, manual, atol=1e-5)

# 3) running mean/var update across batches
bn = BatchNorm1d(4).train(); rm0, rv0 = bn.running_mean.clone(), bn.running_var.clone()
bn(torch.randn(128, 4) * 3 + 2)
assert not torch.allclose(bn.running_mean, rm0) and not torch.allclose(bn.running_var, rv0)

# 4) KEY: numerical match to torch.nn.BatchNorm1d
mine = BatchNorm1d(16); ref = nn.BatchNorm1d(16, eps=1e-5, momentum=0.1)
with torch.no_grad(): ref.weight.copy_(mine.gamma); ref.bias.copy_(mine.beta)
mine.train(); ref.train(); xi = torch.randn(64, 16) * 2.5 - 1.0
assert torch.allclose(mine(xi), ref(xi), atol=1e-5)
assert torch.allclose(mine.running_mean, ref.running_mean, atol=1e-5)
assert torch.allclose(mine.running_var, ref.running_var, atol=1e-5)
mine.eval(); ref.eval(); xe = torch.randn(64, 16)
assert torch.allclose(mine(xe), ref(xe), atol=1e-5)

# 5) gradient flows to gamma and beta
bn = BatchNorm1d(8).train(); bn(torch.randn(32, 8)).sum().backward()
assert bn.gamma.grad.abs().sum() > 0 and bn.beta.grad.abs().sum() > 0

# 6) gamma/beta scale and shift the normalized output
bn = BatchNorm1d(4).train()
with torch.no_grad(): bn.gamma.fill_(2.0); bn.beta.fill_(5.0)
yo = bn(torch.randn(2048, 4))
assert torch.allclose(yo.mean(0), torch.full((4,), 5.0), atol=1e-3)
assert torch.allclose(yo.var(0, unbiased=False), torch.full((4,), 4.0), atol=5e-2)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class BatchNorm1dSolution(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)        # biased var for normalization
            with torch.no_grad():
                n = x.shape[0]
                unbiased_var = var * n / (n - 1)      # unbiased var for running stat
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * unbiased_var)
        else:
            mean = self.running_mean
            var = self.running_var
        xhat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta